# Calibration Notebook

Orchestrates PM and TM engines across the test set, buckets predicted probabilities,
computes actual occurrence rates, and derives USE/SOFT/IGNORE thresholds.

In [1]:
import sys
sys.path.insert(0, r'C:/ClaudeCode')

import pandas as pd
import numpy as np
from datetime import datetime
from models.pm_engine import predict as pm_predict
from models.tm_engine import predict as tm_predict

LOG_FILE = r'C:/ClaudeCode/research/04. backtest_analysis/backtest_analysis.txt'
TRAIN_END = '2024-12-31'
TEST_START = '2025-01-01'

print('Imports OK')

Imports OK


In [2]:
# ── Run PM predictions across test set dates ──────────────────
from models.feature_prep import build_pm_weekly_panel

weekly = build_pm_weekly_panel()
test_weeks = weekly[weekly['week_end_date'] > TRAIN_END].copy()

pm_results = []
for _, row in test_weeks.iterrows():
    dt = row['week_end_date']
    shape = row['shape']
    try:
        pred = pm_predict(dt, shape=shape)
        pred['actual_persists_4w'] = row.get('persists_4w', np.nan)
        pred['actual_persists_12w'] = row.get('persists_12w', np.nan)
        pm_results.append(pred)
    except Exception as e:
        print(f'PM error at {dt.date()}: {e}')

pm_df = pd.DataFrame(pm_results)
print(f'PM predictions: {len(pm_df)} rows')
pm_df.head()

PM predictions: 78 rows


,shape,shape_name,date,predicted_persists_4w,predicted_persists_12w,features_used,method,actual_persists_4w,actual_persists_12w
0,0.0,Contango,2025-01-03,0.809084,0.593079,"[usd_myr, enso_oni]",lr,1.0,1.0
1,0.0,Contango,2025-01-10,0.813073,0.587134,"[usd_myr, enso_oni]",lr,1.0,1.0
2,0.0,Contango,2025-01-17,0.813900,0.585885,"[usd_myr, enso_oni]",lr,0.0,1.0
3,0.0,Contango,2025-01-24,0.803032,0.601871,"[usd_myr, enso_oni]",lr,1.0,0.0
4,0.0,Contango,2025-01-31,0.795756,0.612092,"[usd_myr, enso_oni]",lr,1.0,1.0


In [3]:
# ── Run TM predictions across test set dates ──────────────────
from models.feature_prep import build_tm_daily_panel

daily = build_tm_daily_panel()
test_daily = daily[daily['date'] > TRAIN_END].dropna(
    subset=['target_1w', 'target_2w']).copy()

tm_results = []
for _, row in test_daily.iterrows():
    dt = row['date']
    for hz in ['1w', '2w']:
        try:
            pred = tm_predict(dt, horizon=hz)
            target_col = 'target_1w' if hz == '1w' else 'target_2w'
            pred['actual_next_shape'] = row[target_col]
            pred['correct'] = pred.get('top1_shape') == row[target_col]
            tm_results.append(pred)
        except Exception as e:
            print(f'TM error at {dt.date()} {hz}: {e}')

tm_df = pd.DataFrame(tm_results)
print(f'TM predictions: {len(tm_df)} rows')
tm_df.head()

TM predictions: 700 rows


,current_shape,date,horizon,top1_shape,top1_prob,all_probs,ruled_out,top2_shape,top2_prob,actual_next_shape,correct
0,0.0,2025-01-02,1w,0.0,0.4550,"{'0.0': 0.455, '0.1': 0.3914, '0.2': 0.081, '1...",[2],0.1,0.3914,0.0,True
1,0.0,2025-01-02,2w,0.0,0.8720,"{'0.0': 0.872, '0.1': 0.1019, '0.2': 0.0145, '...","[0.2, 1, 2]",0.1,0.1019,0.0,True
2,0.0,2025-01-03,1w,0.0,0.4316,"{'0.0': 0.4316, '0.1': 0.3944, '1': 0.0934, '0...",[2],0.1,0.3944,0.0,True
3,0.0,2025-01-03,2w,0.0,0.8817,"{'0.0': 0.8817, '0.1': 0.0921, '0.2': 0.0148, ...","[0.2, 1, 2]",0.1,0.0921,0.0,True
4,0.0,2025-01-06,1w,0.0,0.4282,"{'0.0': 0.4282, '0.1': 0.3835, '1': 0.1078, '0...",[2],0.1,0.3835,0.0,True


In [4]:
# ── Bucket analysis + calibration ─────────────────────────────
output_lines = [
    f'CALIBRATION ANALYSIS — {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    '=' * 70,
]

# TM calibration: bucket predicted top-1 probability
for hz in ['1w', '2w']:
    sub = tm_df[tm_df['horizon'] == hz].copy()
    if len(sub) == 0:
        continue

    output_lines.append(f'\n--- TM {hz} CALIBRATION ---')
    output_lines.append(f'Total predictions: {len(sub)}')
    output_lines.append(f'Overall top-1 accuracy: {sub["correct"].mean()*100:.1f}%')

    # Bucket by top1_prob
    bins = [0, 0.3, 0.5, 0.7, 1.01]
    labels = ['<30%', '30-50%', '50-70%', '70%+']
    sub['prob_bucket'] = pd.cut(sub['top1_prob'], bins=bins, labels=labels)

    output_lines.append(f'\n  {"Bucket":<10} {"N":>5} {"Accuracy":>10} {"CI Note":>10}')
    output_lines.append(f'  {"-"*40}')
    for bucket in labels:
        bucket_data = sub[sub['prob_bucket'] == bucket]
        n = len(bucket_data)
        if n == 0:
            continue
        acc = bucket_data['correct'].mean() * 100
        ci_note = 'wide CI' if n < 10 else ''
        output_lines.append(f'  {bucket:<10} {n:>5} {acc:>9.1f}% {ci_note:>10}')

    # Per current_shape breakdown
    output_lines.append(f'\n  Per-shape accuracy:')
    for shape in sorted(sub['current_shape'].unique()):
        shape_sub = sub[sub['current_shape'] == shape]
        acc = shape_sub['correct'].mean() * 100
        output_lines.append(
            f'    Shape {shape}: {acc:.1f}% (n={len(shape_sub)})')

# PM calibration
output_lines.append(f'\n--- PM CALIBRATION ---')
for hz in [4, 12]:
    pred_col = f'predicted_persists_{hz}w'
    act_col = f'actual_persists_{hz}w'
    sub = pm_df.dropna(subset=[act_col]).copy()
    if len(sub) == 0:
        continue

    sub['pred_binary'] = (sub[pred_col] >= 0.5).astype(int)
    sub['actual_binary'] = sub[act_col].astype(int)
    acc = (sub['pred_binary'] == sub['actual_binary']).mean() * 100
    output_lines.append(
        f'  {hz}w: accuracy={acc:.1f}% (n={len(sub)})')

# Derive thresholds
output_lines.append(f'\n--- DERIVED THRESHOLDS ---')
for hz in ['1w', '2w']:
    sub = tm_df[tm_df['horizon'] == hz].copy()
    if len(sub) == 0:
        continue

    output_lines.append(f'\n  {hz} horizon:')
    for shape in sorted(sub['current_shape'].unique()):
        shape_sub = sub[sub['current_shape'] == shape]
        if len(shape_sub) < 5:
            continue

        # For each possible next shape, classify as USE/SOFT/IGNORE
        all_probs_records = []
        for _, row in shape_sub.iterrows():
            ap = row.get('all_probs', {})
            if isinstance(ap, dict):
                for next_s, prob in ap.items():
                    all_probs_records.append({
                        'next_shape': next_s,
                        'prob': prob,
                        'actual': row['actual_next_shape'],
                        'occurred': int(row['actual_next_shape'] == next_s),
                    })

        if not all_probs_records:
            continue

        prob_df = pd.DataFrame(all_probs_records)
        output_lines.append(f'    Shape {shape}:')

        for next_s in sorted(prob_df['next_shape'].unique()):
            ns = prob_df[prob_df['next_shape'] == next_s]
            if len(ns) < 5:
                continue
            median_prob = ns['prob'].median()
            high = ns[ns['prob'] > median_prob]
            low = ns[ns['prob'] <= median_prob]
            if len(high) < 3 or len(low) < 3:
                continue
            gap = high['occurred'].mean()*100 - low['occurred'].mean()*100
            rating = 'USE' if gap > 5 else ('SOFT' if gap > 2 else 'IGNORE')
            output_lines.append(
                f'      -> {next_s}: gap={gap:+.1f}pp  rating={rating}')

# Write to log
content = '\n'.join(output_lines)
print(content)

with open(LOG_FILE, 'a', encoding='utf-8') as f:
    f.write('\n' + content + '\n')
print(f'\n[Written to {LOG_FILE}]')

CALIBRATION ANALYSIS — 2026-07-10 08:55

--- TM 1w CALIBRATION ---
Total predictions: 350
Overall top-1 accuracy: 46.6%

  Bucket         N   Accuracy    CI Note
  ----------------------------------------
  <30%          22      22.7%           
  30-50%       245      41.2%           
  50-70%        68      63.2%           
  70%+          15      93.3%           

  Per-shape accuracy:
    Shape 0.0: 55.0% (n=80)
    Shape 0.1: 44.0% (n=25)
    Shape 0.2: 14.3% (n=7)
    Shape 1: 57.7% (n=111)
    Shape 2: 33.9% (n=127)

--- TM 2w CALIBRATION ---
Total predictions: 350
Overall top-1 accuracy: 60.9%

  Bucket         N   Accuracy    CI Note
  ----------------------------------------
  30-50%        96      54.2%           
  50-70%       106      57.5%           
  70%+         148      67.6%           

  Per-shape accuracy:
    Shape 0.0: 76.2% (n=80)
    Shape 0.1: 36.0% (n=25)
    Shape 0.2: 0.0% (n=7)
    Shape 1: 56.8% (n=111)
    Shape 2: 63.0% (n=127)

--- PM CALIBRATION ---
